<a href="https://colab.research.google.com/github/Abdifatah2023/Abdifatah2023/blob/main/workingProgress.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Milestone 1: Download and prepare data

In [189]:
!git clone https://github.com/bhavanachem/RedditBias.git

fatal: destination path 'RedditBias' already exists and is not an empty directory.


In [190]:
!pip install pandas scikit-learn -q

In [191]:
import pandas as pd
from sklearn.model_selection import train_test_split
from datasets import load_dataset, Dataset

In [192]:
file1 = "RedditBias/data/gender/reddit_comments_gender_female_processed_phrase_annotated.csv"
file2 = "RedditBias/data/orientation/reddit_comments_orientation_lgbtq_processed_phrase_annotated.csv"
file3 = "RedditBias/data/race/reddit_comments_race_black_processed_phrase_annotated.csv"
file4 = "RedditBias/data/religion1/reddit_comments_religion1_jews_processed_phrase_annotated.csv"
file5 = "RedditBias/data/religion2/reddit_comments_religion2_muslims_processed_phrase_annotated.csv"

print("Loading individual datasets...")
df1 = pd.read_csv(file1)
df2 = pd.read_csv(file2)
df3 = pd.read_csv(file3)
df4 = pd.read_csv(file4, encoding='latin-1')
df5 = pd.read_csv(file5)

print("Working with: ")
print(file1)
print(file2)
print(file3)
print(file4)
print(file5)

Loading individual datasets...
Working with: 
RedditBias/data/gender/reddit_comments_gender_female_processed_phrase_annotated.csv
RedditBias/data/orientation/reddit_comments_orientation_lgbtq_processed_phrase_annotated.csv
RedditBias/data/race/reddit_comments_race_black_processed_phrase_annotated.csv
RedditBias/data/religion1/reddit_comments_religion1_jews_processed_phrase_annotated.csv
RedditBias/data/religion2/reddit_comments_religion2_muslims_processed_phrase_annotated.csv


In [193]:
# Combine datasets
df = pd.concat([df1, df2, df3, df4, df5], ignore_index=True)

In [194]:
df

,attribute_in_window,comment,phrase,bias_sent,bias_phrase,Unnamed: 5,id
0,True,im sorry this is happening for you. at least y...,least you know whats happening now. narcissist...,0,0.0,NaN,NaN
1,True,their mothers are in on the joke art project,their mothers are in on the joke art project,1,1.0,NaN,NaN
2,True,i feel like this is some deep artistic dance t...,artistic dance thats trying to highlight how m...,0,0.0,NaN,NaN
3,True,or biology. over protective and smothering mot...,or biology. over protective and smothering mot...,0,0.0,NaN,NaN
4,True,writing poetry to a girl is the fastest way to...,writing poetry to a girl is the fastest way to...,0,0.0,NaN,NaN
...,...,...,...,...,...,...,...
28126,True,false comparison not every muslim is violent. ...,false comparison not every muslim is violent. ...,NaN,NaN,NaN,dlre79o
28127,True,if every nazi is violent because they believe ...,believe in a violent ideology then every musli...,NaN,NaN,NaN,dlreadd
28128,True,guys i dont think they know just how stupid th...,how stupid they are not every muslim is viole...,NaN,NaN,NaN,dlreflm
28129,True,not every muslim is violent not every german i...,not every muslim is violent not every german i...,NaN,NaN,NaN,disuk3y


In [195]:
df.columns

Index(['attribute_in_window', 'comment', 'phrase', 'bias_sent', 'bias_phrase',
       'Unnamed: 5', 'id'],
      dtype='object')

In [196]:
# Select only the columns we need
new_df = df[['comment', 'bias_sent']]
new_df

,comment,bias_sent
0,im sorry this is happening for you. at least y...,0
1,their mothers are in on the joke art project,1
2,i feel like this is some deep artistic dance t...,0
3,or biology. over protective and smothering mot...,0
4,writing poetry to a girl is the fastest way to...,0
...,...,...
28126,false comparison not every muslim is violent. ...,NaN
28127,if every nazi is violent because they believe ...,NaN
28128,guys i dont think they know just how stupid th...,NaN
28129,not every muslim is violent not every german i...,NaN


In [197]:
# Convert the column to numeric, coercing errors will turn non-numeric values into NaN
new_df['bias_sent'] = pd.to_numeric(df['bias_sent'], errors='coerce')
print(new_df['bias_sent'].value_counts().sort_index())

bias_sent
0.0    4900
1.0    6592
2.0      24
Name: count, dtype: int64


/tmp/ipython-input-2539419507.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['bias_sent'] = pd.to_numeric(df['bias_sent'], errors='coerce')


In [198]:
# split into labeled and unlabeled data columns
df_labeled = new_df.dropna(subset=['bias_sent'])
df_unlabeled = new_df[new_df['bias_sent'].isna()]

In [199]:
print(df_labeled['bias_sent'].value_counts().sort_index())

bias_sent
0.0    4900
1.0    6592
2.0      24
Name: count, dtype: int64


In [200]:
df_labeled['bias_sent'] = df_labeled['bias_sent'].astype(int)
df_labeled = df_labeled[df_labeled['bias_sent'].isin([0, 1])].copy()
df_labeled.dtypes

/tmp/ipython-input-2749396818.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_labeled['bias_sent'] = df_labeled['bias_sent'].astype(int)


,0
comment,object
bias_sent,int64


In [201]:
df_labeled['comment'] = df_labeled['comment'].astype("string")
df_labeled = df_labeled[df_labeled['comment'].str.len() > 2].reset_index(drop=True)
df_labeled.dtypes

,0
comment,string[python]
bias_sent,int64


In [202]:
print(df_labeled['bias_sent'].value_counts().sort_index())

bias_sent
0    4900
1    6592
Name: count, dtype: int64


In [203]:
df_labeled.to_csv("redditbias_labeled.csv", index=False)
df_unlabeled.to_csv("redditbias_unlabeled.csv", index=False)

In [204]:
# Define text and label columns
TEXT_COLUMN = 'comment'  # The main text we'll classify
LABEL_COLUMN = 'bias_sent'  # Using sentence-level bias as our target

print("Using", TEXT_COLUMN, "as input text")
print("Using", LABEL_COLUMN, "as target label")

Using comment as input text
Using bias_sent as target label


In [205]:
print("First 10 rows: ")
df_labeled.head(10)

First 10 rows: 


,comment,bias_sent
0,im sorry this is happening for you. at least y...,0
1,their mothers are in on the joke art project,1
2,i feel like this is some deep artistic dance t...,0
3,or biology. over protective and smothering mot...,0
4,writing poetry to a girl is the fastest way to...,0
5,your wife is sexual poetry in motion,0
6,top banana horse girl is like poetry,0
7,my wife is a poet and loves poetry. it is amaz...,1
8,this girl is pure poetry.,0
9,not to be confused with the one where the girl...,1


In [206]:
# Print the shape and total number of samples in the combined dataset
print("Dataset shape: ", df_labeled.shape)
print("Total samples: ", len(df_labeled))

Dataset shape:  (11492, 2)
Total samples:  11492


In [207]:
print("Column names: ", df_labeled.columns.tolist())
print("Column data types: ", df_labeled.dtypes)

Column names:  ['comment', 'bias_sent']
Column data types:  comment      string[python]
bias_sent             int64
dtype: object


#### DATASET IS READY FOR SPLITS
##### Will use the labeled dataset for training the model and the unlabeled dataset for testing and evaluation, so that we can later use those predicted values to label the unlabeled values in that dataset.

In [208]:
# load labeled df/csv
df = pd.read_csv('redditbias_labeled.csv')
df

,comment,bias_sent
0,im sorry this is happening for you. at least y...,0
1,their mothers are in on the joke art project,1
2,i feel like this is some deep artistic dance t...,0
3,or biology. over protective and smothering mot...,0
4,writing poetry to a girl is the fastest way to...,0
...,...,...
11487,saying that a muslim is violent because they d...,0
11488,funny that a muslim is talking about other rel...,1
11489,a good muslim is a violent racist homophobic ...,1
11490,yeah no muslim is violent,0


In [209]:
# Create initial 80% train, 20% validation split
train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df['bias_sent']  # Stratify to keep label distribution
)
print(len(train_df), len(val_df))

9193 2299


In [210]:
# Save the splits
train_df.to_csv("train.csv", index=False)
val_df.to_csv("validation.csv", index=False)

print("Data splitting complete.")
print(f"Train set size: {len(train_df)}")
print(f"Validation set size: {len(val_df)}")

Data splitting complete.
Train set size: 9193
Validation set size: 2299


# Milestone 2: Finetune base model

In [211]:
!pip install transformers datasets evaluate accelerate -q

In [212]:
import torch
import numpy as np
import evaluate
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
from sklearn.metrics import accuracy_score

In [213]:
MODEL_NAME_BERT = 'bert-base-uncased'

In [214]:
# 1. Load the data splits
print("Loading data splits...")
data_files = {
    "train": "train.csv",
    "validation": "validation.csv",
}

# Load from file and rename the target column
raw_datasets = load_dataset('csv', data_files=data_files)
raw_datasets = raw_datasets.rename_column(LABEL_COLUMN, "labels")

Loading data splits...


Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

In [215]:
# 2. Load Tokenizer
print(f"Loading tokenizer for {MODEL_NAME_BERT}...")
tokenizer_bert = AutoTokenizer.from_pretrained(MODEL_NAME_BERT)

Loading tokenizer for bert-base-uncased...


In [216]:
# 3. Define Preprocessing Function (Tokenization)
def tokenize_function_bert(examples):
    return tokenizer_bert(
        examples[TEXT_COLUMN],
        padding='max_length',
        truncation=True,
        max_length=128
    )

In [217]:
# 4. Apply Tokenization
print("Tokenizing datasets...")
# We use .map() to apply the tokenization to the entire dataset
tokenized_datasets_bert = raw_datasets.map(tokenize_function_bert, batched=True)

Tokenizing datasets...


Map:   0%|          | 0/9193 [00:00<?, ? examples/s]

Map:   0%|          | 0/2299 [00:00<?, ? examples/s]

In [218]:
# 5. Load the Model
print(f"Loading model {MODEL_NAME_BERT}...")
model_bert = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME_BERT,
    num_labels=2
)

Loading model bert-base-uncased...


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [219]:
# 6. Define Evaluation Metrics
# We use accuracy as a simple metric.
metric_accuracy = evaluate.load("accuracy")

def compute_metrics_bert(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric_accuracy.compute(predictions=predictions, references=labels)

In [220]:
# 7. Define Training Arguments
training_args_bert = TrainingArguments(
    output_dir="./results_bert",
    eval_strategy="epoch",
    logging_steps=100,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="none",
)

In [221]:
# 8. Initialize the Data Collator
print("Initializing Data Collator for dynamic padding...")
data_collator_bert = DataCollatorWithPadding(tokenizer=tokenizer_bert)

Initializing Data Collator for dynamic padding...


In [222]:
# 9. Initialize the HuggingFace Trainer
# This class handles all the training and evaluation logic.
trainer_bert = Trainer(
    model=model_bert,
    args=training_args_bert,
    train_dataset=tokenized_datasets_bert["train"],
    eval_dataset=tokenized_datasets_bert["validation"],
    data_collator=data_collator_bert,
    compute_metrics=compute_metrics_bert
)

In [223]:
# 10. Start Finetuning
print("Starting BERT model finetuning...")
trainer_bert.train()

print("BERT Training complete.")

Starting BERT model finetuning...


Epoch,Training Loss,Validation Loss,Accuracy
1,0.458600,0.451345,0.795998
2,0.344300,0.431386,0.824706
3,0.212500,0.512836,0.817747


BERT Training complete.


In [224]:
# Pseudo-labeling

def predict_proba(trainer, dataset, tokenizer, batch_size=64):
    # dataset is a HuggingFace Dataset with 'text'
    ds = dataset.map(lambda x: tokenizer(x['comment'], truncation=True, padding='max_length', max_length=128), batched=True)
    preds = trainer.predict(ds, metric_key_prefix="predict")
    logits = preds.predictions
    probs = torch.softmax(torch.tensor(logits), dim=1).numpy()
    return probs

# load unlabeled CSV
df_unlabeled = pd.read_csv("redditbias_unlabeled.csv")
unlabeled_ds = Dataset.from_pandas(df_unlabeled[['comment']].reset_index(drop=True))

# pseudo-label loop
threshold = 0.95
max_iters = 3

for it in range(max_iters):
    print(f"Pseudo-label iteration {it+1}")
    probs = predict_proba(trainer_bert, unlabeled_ds, tokenizer_bert)
    preds = np.argmax(probs, axis=1)
    max_probs = probs.max(axis=1)

    high_conf_mask = max_probs >= threshold
    n_high = high_conf_mask.sum()
    print(f"Found {n_high} high-confidence unlabeled samples (threshold {threshold})")
    if n_high == 0:
        break

    selected_texts = df_unlabeled.loc[high_conf_mask, 'comment'].tolist()
    selected_labels = preds[high_conf_mask].tolist()

    # create dataframe and append to labeled dataset
    df_new = pd.DataFrame({'comment': selected_texts, 'bias_sent': selected_labels})
    df_new.to_csv(f"pseudo_labels_iter{it+1}.csv", index=False)

    # update labeled and unlabeled pools
    df_labeled = pd.concat([pd.read_csv("redditbias_labeled.csv"), df_new], ignore_index=True)
    df_labeled.to_csv("redditbias_labeled_expanded.csv", index=False)

    # remove from unlabeled
    df_unlabeled = df_unlabeled.loc[~high_conf_mask].reset_index(drop=True)
    df_unlabeled.to_csv("redditbias_unlabeled_remaining.csv", index=False)


    # Rebuild HF datasets, retrain (or continue)
    train_df, val_df = train_test_split(df_labeled, test_size=0.2, stratify=df_labeled['bias_sent'], random_state=42)
    train_ds = Dataset.from_pandas(train_df[['comment','bias_sent']].rename(columns={'bias_sent':'labels'}))
    val_ds = Dataset.from_pandas(val_df[['comment','bias_sent']].rename(columns={'bias_sent':'labels'}))

    # apply tokenization
    train_ds = train_ds.map(tokenize_function_bert, batched=True)
    val_ds = val_ds.map(tokenize_function_bert, batched=True)


    # Option A: continue training from current model
    trainer_bert.train_dataset = train_ds
    trainer_bert.eval_dataset = val_ds
    trainer_bert.train()
    trainer_bert.save_model(f"bert-bias-pseudo-iter{it+1}")

    # Recreate unlabeled HF dataset for next iteration
    unlabeled_ds = Dataset.from_pandas(df_unlabeled[['comment']].reset_index(drop=True))


Pseudo-label iteration 1


Map:   0%|          | 0/16615 [00:00<?, ? examples/s]

Found 5050 high-confidence unlabeled samples (threshold 0.95)


Map:   0%|          | 0/13233 [00:00<?, ? examples/s]

Map:   0%|          | 0/3309 [00:00<?, ? examples/s]

...

Map:   0%|          | 0/6488 [00:00<?, ? examples/s]

Found 4941 high-confidence unlabeled samples (threshold 0.95)


Map:   0%|          | 0/13146 [00:00<?, ? examples/s]

Map:   0%|          | 0/3287 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Accuracy
1,0.186200,0.447958,0.869486
2,0.103800,0.609016,0.880742
3,0.067300,0.658418,0.885002


In [225]:
train_ds.to_csv('train.csv', index=False)
val_ds.to_csv('validation.csv', index=False)

Creating CSV from Arrow format:   0%|          | 0/14 [00:00<?, ?ba/s]

Creating CSV from Arrow format:   0%|          | 0/4 [00:00<?, ?ba/s]

4592811

In [226]:
# check class distribution (all labeled data)
df_labeled = pd.read_csv('redditbias_labeled.csv')
df_labeled['bias_sent'].value_counts(normalize=True)

,proportion
bias_sent,
1,0.573616
0,0.426384


In [227]:
# Save the final model
trainer_bert.save_model("./final_model_bert")
tokenizer_bert.save_pretrained("./final_model_bert")
print("Final BERT model saved to ./final_model_bert")

Final BERT model saved to ./final_model_bert


# Milestone 3: Test/evaluate additional datasets' bias

In [228]:
from sklearn.metrics import f1_score, precision_score, recall_score

In [229]:
print("Starting Milestone 3: CrowS-Pairs Evaluation")

Starting Milestone 3: CrowS-Pairs Evaluation


In [230]:
# 1. Load and filter CrowS-Pairs Dataset
print("Loading and filtering CrowS-Pairs dataset...")

crows_df = pd.read_csv("crows_pairs_anonymized.csv")

relevant_types = ['gender', 'race', 'religion', 'sexual_orientation']
crows_filtered_df = crows_df[crows_df['bias_type'].isin(relevant_types)].copy()

Loading and filtering CrowS-Pairs dataset...


In [231]:
# 2. Adapt for Binary Classification
df_more = crows_filtered_df.rename(columns={'sent_more': 'comment'}).copy()
df_more['labels'] = 1 # Stereotypical sentence = Biased
df_less = crows_filtered_df.rename(columns={'sent_less': 'comment'}).copy()
df_less['labels'] = 0 # Less stereotypical sentence = Not Biased

crows_eval_df = pd.concat([df_more, df_less], ignore_index=True)
crows_eval_df = crows_eval_df[['comment', 'labels']].copy()

crows_eval_dataset = Dataset.from_pandas(crows_eval_df)

In [232]:
crows_eval_dataset

Dataset({
    features: ['comment', 'labels'],
    num_rows: 734
})

In [233]:
# 3. Tokenize with BERT tokenizer
tokenized_crows = crows_eval_dataset.map(tokenize_function_bert, batched=True)
tokenized_crows = tokenized_crows.remove_columns([
    col for col in tokenized_crows.column_names
    if col not in ['input_ids', 'attention_mask', 'labels']
])

Map:   0%|          | 0/734 [00:00<?, ? examples/s]

In [234]:
# 4. Evaluate with the Finetuned BERT Model
print("Evaluating BERT model on CrowS-Pairs dataset...")
crows_predictions = trainer_bert.predict(tokenized_crows)
pred_labels = np.argmax(crows_predictions.predictions, axis=-1)

crows_eval_df['predicted_label'] = pred_labels

Evaluating BERT model on CrowS-Pairs dataset...


In [235]:
# Analyze overall evaluation results (no bias categories)
print("CrowS-Pairs Overall Metrics (BERT Model)")

# Compute global metrics
overall_accuracy = accuracy_score(crows_eval_df['labels'], crows_eval_df['predicted_label'])
overall_f1 = f1_score(crows_eval_df['labels'], crows_eval_df['predicted_label'], average='binary', zero_division=0)

results = {
    "accuracy": round(overall_accuracy, 4),
    "f1_score": round(overall_f1, 4),
    "count": len(crows_eval_df)
}

print(pd.DataFrame([results]))


CrowS-Pairs Overall Metrics (BERT Model)
   accuracy  f1_score  count
0    0.5136    0.5182    734


# Milestone 4: Improve the finetuned model

In [236]:
print("Starting Milestone 4: RoBERTa Finetuning")

Starting Milestone 4: RoBERTa Finetuning


In [237]:
# 1. Define enhanced metrics function (F1, Precision, Recall)
def compute_metrics_full(eval_pred):
    """Now we compute accuracy, F1, precision, and recall for binary classification."""
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    accuracy = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average='binary', zero_division=0)
    precision = precision_score(labels, predictions, average='binary', zero_division=0)
    recall = recall_score(labels, predictions, average='binary', zero_division=0)

    return {
        'accuracy': accuracy,
        'f1': f1,
        'precision': precision,
        'recall': recall,
    }

In [238]:
# 2: Try out RoBERTa model
MODEL_NAME_ROBERTA = "roberta-base"

# Load Tokenizer for RoBERTa
print("Loading tokenizer for RoBERTa...")
tokenizer_roberta = AutoTokenizer.from_pretrained(MODEL_NAME_ROBERTA)

# Define RoBERTa Tokenization Function
def tokenize_function_roberta(examples):
    return tokenizer_roberta(
        examples[TEXT_COLUMN],
        return_token_type_ids=False,
        padding='max_length',
        max_length=128,
        truncation=True
    )

Loading tokenizer for RoBERTa...


In [239]:
# 1. Load the redditbias_labeled_expanded.csv file
df_roberta_prep = pd.read_csv('redditbias_labeled_expanded.csv')

# 4. Split df_roberta_prep into training and validation sets
train_df, val_df = train_test_split(
    df_roberta_prep,
    test_size=0.2,
    random_state=42,
    stratify=df_roberta_prep['bias_sent']
)

# 5. Save train_df to 'train.csv'
train_df.to_csv('train.csv', index=False)

# 6. Save val_df to 'validation.csv'
val_df.to_csv('validation.csv', index=False)

In [240]:
# 2. Rename 'bias_sent' → 'labels'
train_df = train_df.rename(columns={LABEL_COLUMN: "labels"})
val_df = val_df.rename(columns={LABEL_COLUMN: "labels"})

In [241]:
train_ds = Dataset.from_pandas(train_df[["comment", "labels"]])
val_ds   = Dataset.from_pandas(val_df[["comment", "labels"]])

In [242]:
# 4. Apply Tokenization
print("Tokenizing datasets...")
# We use .map() to apply the tokenization to the entire dataset
train_ds = train_ds.map(tokenize_function_roberta, batched=True)
val_ds = val_ds.map(tokenize_function_roberta, batched=True)


Tokenizing datasets...


Map:   0%|          | 0/13146 [00:00<?, ? examples/s]

Map:   0%|          | 0/3287 [00:00<?, ? examples/s]

In [243]:
# Load the RoBERTa model
print("Loading RoBERTa model...")
model_roberta = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME_ROBERTA,
    num_labels=2
)

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Loading RoBERTa model...


In [244]:
# 3. Define RoBERTa Training Arguments
# TUNE: Lower learning rate and higher weight decay than BERT baseline
training_args_roberta = TrainingArguments(
    output_dir="./results_roberta",
    eval_strategy="epoch",
    logging_steps=100,
    learning_rate=1e-5, # TUNE: Lowered from 2e-5 (BERT)
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.05, # TUNE: Increased from 0.01 (BERT)
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="none",
)

In [245]:
# Initiliaze the Data Collator
data_collator_roberta = DataCollatorWithPadding(tokenizer=tokenizer_roberta)

# Initiliaze the HuggingFace Trainer
trainer_roberta = Trainer(
    model=model_roberta,
    args=training_args_roberta,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer_roberta,
    data_collator=data_collator_roberta,
    compute_metrics=compute_metrics_full, # Use the enhanced metric function
)

/tmp/ipython-input-1591929142.py:5: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer_roberta = Trainer(


In [246]:
# 4. Start Finetuning RoBERTa
print("Starting RoBERTa model finetuning...")

trainer_roberta.train()
print("RoBERTa Training complete.")

Starting RoBERTa model finetuning...


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.452800,0.462274,0.798905,0.816440,0.823068,0.809917
2,0.365200,0.453622,0.816854,0.832031,0.842849,0.821488
3,0.305300,0.474056,0.822939,0.837883,0.847324,0.828650


RoBERTa Training complete.


In [247]:
# Save the final RoBERTa model
trainer_roberta.save_model("./final_model_roberta")
tokenizer_roberta.save_pretrained("./final_model_roberta")
print("Final RoBERTa model saved to ./final_model_roberta")

Final RoBERTa model saved to ./final_model_roberta


# Extra: RoBERTa Evaluation on External Dataset (CrowS-Pairs)

In [248]:
print("Evaluating RoBERTa model on CrowS-Pairs dataset...")

Evaluating RoBERTa model on CrowS-Pairs dataset...


In [249]:
# 1. Get predictions from the trained RoBERTa trainer
crows_predictions_roberta = trainer_roberta.predict(tokenized_crows)
pred_labels_roberta = np.argmax(crows_predictions_roberta.predictions, axis=-1)

In [250]:
# 2. Add predictions to the original evaluation DataFrame
crows_eval_df['predicted_label_roberta'] = pred_labels_roberta

In [251]:
# 3. Calculate and display overall metrics (using the full metrics function)
roberta_external_results = compute_metrics_full(
    (crows_predictions_roberta.predictions, crows_eval_df['labels'].values)
)

print("RoBERTa External Dataset Results:")
print(roberta_external_results)

RoBERTa External Dataset Results:
{'accuracy': 0.49727520435967304, 'f1': 0.4533333333333333, 'precision': 0.4967532467532468, 'recall': 0.41689373297002724}


In [252]:
# Analyze overall evaluation results (no bias categories)
print("CrowS-Pairs Overall Metrics (RoBERTa Model)")

# Compute global metrics
overall_accuracy = accuracy_score(crows_eval_df['labels'], crows_eval_df['predicted_label'])
overall_f1 = f1_score(crows_eval_df['labels'], crows_eval_df['predicted_label'], average='binary', zero_division=0)

results = {
    "accuracy": round(overall_accuracy, 4),
    "f1_score": round(overall_f1, 4),
    "count": len(crows_eval_df)
}

print(pd.DataFrame([results]))


CrowS-Pairs Overall Metrics (RoBERTa Model)
   accuracy  f1_score  count
0    0.5136    0.5182    734
